# Bonus Project: African Wildlife Object Detection (YOLO11n)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/csci1409-object-detection/blob/main/notebooks/object_detection_colab.ipynb)

This notebook runs the full project from a **fresh Colab runtime**:
1. install pinned dependencies  2. download + check the data  3. train the baseline
4. train the controlled experiment  5. compare on validation  6. evaluate the selected run on test
7. show predictions and failure cases  8. package the results.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
Full run takes roughly 40-60 min on a free T4. Set `QUICK_RUN = True` for a 1-epoch smoke test (also works on CPU / local Jupyter).

## 0. Settings

In [ ]:
REPO_URL = "https://github.com/eskendir2006/csci1409-object-detection.git"  # <-- your repo
QUICK_RUN = False   # True = 1 epoch on 5% of the training data, only to check that everything runs

SMOKE_ARGS = ["--epochs", "1", "--fraction", "0.05"] if QUICK_RUN else []
SUFFIX = "_smoke" if QUICK_RUN else ""
BASELINE, EXPERIMENT = "baseline_640" + SUFFIX, "exp_imgsz416" + SUFFIX

## 1. Get the code and install pinned dependencies

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/csci1409-object-detection").exists():
        !git clone {REPO_URL} /content/csci1409-object-detection
    os.chdir("/content/csci1409-object-detection")
elif not Path("scripts").exists():          # local Jupyter: notebook lives in notebooks/
    os.chdir("..")
print("Working directory:", os.getcwd())

import subprocess

def run(script, *args):
    """Run a project script with THIS kernel's Python, stream its output, stop on failure."""
    cmd = [sys.executable, "-u", script, *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    env = {**os.environ, "MPLBACKEND": "Agg", "PYTHONIOENCODING": "utf-8"}
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace", env=env)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"{script} failed with exit code {p.returncode}")

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import platform, torch, ultralytics
print("python     ", platform.python_version())
print("torch      ", torch.__version__, "| CUDA:", torch.version.cuda)
print("ultralytics", ultralytics.__version__)
print("GPU        ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

## 2. Prepare and inspect the data
Downloads the Ultralytics *African Wildlife* dataset (YOLO format, 4 classes) into `datasets/`,
finds near-duplicate images with a perceptual hash and moves any duplicate group that spans two
splits into a single split (deterministic, no randomness), checks every label file, and draws
ground-truth boxes.

In [ ]:
run("scripts/prepare_data.py", "--resolve-duplicates")

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(Path("results/dataset/dataset_stats.md").read_text()))
display(Image("results/dataset/gt_samples.png"))
if Path("results/dataset/duplicate_pairs.png").exists():
    display(Image("results/dataset/duplicate_pairs.png", width=600))

## 3. Train the baseline
Fine-tunes COCO-pretrained `yolo11n.pt`. All hyper-parameters are in `configs/baseline.yaml`.
The console log, learning curves and `run_info.json` (versions, GPU, training time, best epoch)
are saved to `results/runs/<run>/`; weights go to `runs/<run>/weights/best.pt`.

In [ ]:
run("scripts/train.py", "--config", "configs/baseline.yaml", *SMOKE_ARGS)

## 4. Controlled experiment: image size 640 -> 416
`configs/exp_imgsz416.yaml` is identical to the baseline except `imgsz`.

**Hypothesis (written before training):** reducing the input size from 640 to 416 px will make
training noticeably faster (about 2.4x fewer pixels per image), but will lower recall and
mAP@0.50 a little, mostly for small or distant animals and crowded groups (e.g. zebra herds),
because these objects cover fewer pixels after downscaling.

In [ ]:
run("scripts/train.py", "--config", "configs/exp_imgsz416.yaml", *SMOKE_ARGS)

## 5. Compare both runs on the validation set
Each run's `best.pt` is evaluated on **val** (conf=0.001, NMS IoU=0.7). The final run is
chosen by validation mAP@0.50. The test set is not used for this decision.

In [ ]:
run("scripts/evaluate.py", "--run", BASELINE, "--split", "val")
run("scripts/evaluate.py", "--run", EXPERIMENT, "--split", "val")
run("scripts/compare_runs.py", "--runs", BASELINE, EXPERIMENT)

In [ ]:
display(Markdown(Path("results/comparison.md").read_text()))
display(Image("results/learning_curves.png"))
display(Image(f"results/runs/{BASELINE}/results.png", width=900))

## 6. Final evaluation on the held-out test set
Only the selected run is evaluated on test, once.

In [ ]:
import json
SELECTED = json.loads(Path("results/selected_run.json").read_text())["selected_run"]
print("Selected run:", SELECTED)
run("scripts/evaluate.py", "--run", SELECTED, "--split", "test")

## 7. Test predictions and failure analysis
Predictions are drawn at conf >= 0.25. A prediction is correct when IoU >= 0.5 with a
same-class ground-truth box. Green dashed = ground truth, blue = correct, red = error.

In [ ]:
run("scripts/analyze_test.py", "--run", SELECTED, "--conf", 0.25)

In [ ]:
print(Path("results/test_analysis/summary.json").read_text())
display(Image("results/test_analysis/predictions_grid.png"))
display(Markdown(Path("results/test_analysis/failures.md").read_text()))
for f in sorted(Path("results/test_analysis").glob("failure_*.png")):
    display(Image(str(f), width=700))

## 8. Package results and the final checkpoint
Downloads `project_outputs.zip` (evidence for the repo) and `best.pt` (upload it to a GitHub
Release; see README).

In [ ]:
import shutil
best = Path(f"runs/{SELECTED}/weights/best.pt")
shutil.copy2(best, "best.pt")
shutil.make_archive("project_outputs", "zip", ".", "results")
print("Created project_outputs.zip and best.pt")
if IN_COLAB:
    from google.colab import files
    files.download("project_outputs.zip")
    files.download("best.pt")

## 9. (Optional) Predict with the released checkpoint
Anyone can run predictions without training: download the released weights and run them on
any image (here: the test split). Run sections 0-2 first so the code and data are present.

In [ ]:
WEIGHTS_URL = "https://github.com/eskendir2006/csci1409-object-detection/releases/download/v1.0/best.pt"

import json
from ultralytics import YOLO
from ultralytics.utils.downloads import safe_download
SELECTED = json.loads(Path("results/selected_run.json").read_text())["selected_run"]
IMGSZ = json.loads(Path(f"results/runs/{SELECTED}/run_info.json").read_text())["hyperparameters"]["imgsz"]
if not Path("released_best.pt").exists():
    safe_download(WEIGHTS_URL, file="released_best.pt")
model = YOLO("released_best.pt")
test_imgs = sorted(Path("datasets/african-wildlife/images/test").glob("*.jpg"))[:3]
for r in model.predict([str(p) for p in test_imgs], imgsz=IMGSZ, conf=0.25):
    r.save(filename=f"pred_{Path(r.path).name}")
    display(Image(f"pred_{Path(r.path).name}", width=500))